# <span style='font-size: 32px;'>ML-Adult-income</span>

Réalisée par ZAHID Zainab 

Import des bibliothèques necessaires

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Créer le dossier images s'il n'existe pas
os.makedirs('../images', exist_ok=True)

Chargement du dataset

In [ ]:
columns = [
    'age', 'workclass', 'fnlwgt', 'education',
    'education_num', 'marital_status', 'occupation',
    'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss',
    'hours_per_week', 'native_country', 'income'
]

# Vérifier si le fichier existe
data_path = '../data/adult.data'
if not os.path.exists(data_path):
    print(f"Erreur: Le fichier {data_path} n'existe pas.")
    print(f"Assurez-vous que le fichier est dans le bon répertoire.")
else:
    df = pd.read_csv(
        data_path,
        names=columns,
        sep=', ',
        engine='python'
    )
    print("Dataset chargé avec succès")
    df.head()

Exploration du dataset

Affichage d'informations

In [ ]:
df.info()

Valeurs manquantes

In [ ]:
df.isnull().sum()

Taille du dataset

In [ ]:
df.shape

Visualisation

Distribution des revenus

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='income')
plt.title("Distribution des revenus")
plt.savefig('../images/income_distribution.png')
plt.show()

Distribution de l'âge

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['age'], bins=30)
plt.title("Distribution de l'âge")
plt.savefig('../images/age_distribution.png')
plt.show()

Heures de travail

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='income', y='hours_per_week')
plt.title("Heures de travail par revenu")
plt.savefig('../images/Heures_travail.png')
plt.show()

Prétraitement

Remplacement des ?

In [ ]:
df.replace('?', np.nan, inplace=True)

Suppression des valeurs nulles

In [ ]:
df.dropna(inplace=True)
print(f"Taille après suppression des valeurs nulles: {df.shape}")

Encodage des colonnes catégorielles

In [ ]:
# Sauvegarder les noms de colonnes avant l'encodage
feature_names = [col for col in df.columns if col != 'income']

encoder = LabelEncoder()

for column in df.columns:
    if df[column].dtype == 'object':
        df[column] = encoder.fit_transform(df[column])

Séparation de X et y

In [ ]:
X = df.drop('income', axis=1)
y = df['income']

Normalisation

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

Train/Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

Modèle 1 : KNN

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

print("Accuracy KNN :", accuracy_score(y_test, y_pred_knn))

Modèle 2 : Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Accuracy Decision Tree :", accuracy_score(y_test, y_pred_dt))

Modèle 3 : Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Accuracy Logistic Regression :", accuracy_score(y_test, y_pred_lr))

Modèle 4 : Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Accuracy Random Forest :", accuracy_score(y_test, y_pred_rf))

Comparaison des modèles

In [ ]:
results = pd.DataFrame({
    'Model': [
        'KNN',
        'Decision Tree',
        'Logistic Regression',
        'Random Forest'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ]
})

results

Matrice de confusion

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig('../images/confusion_matrix_rf.png')
plt.show()

Rapport détaillé

In [ ]:
print(classification_report(y_test, y_pred_rf))

Graphique comparaison modèles

In [ ]:
plt.figure(figsize=(10, 6))
results.set_index('Model').plot(kind='bar')
plt.title("Comparison of Models")
plt.ylabel("Accuracy")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../images/model_comparison.png')
plt.show()

Feature importance (Random Forest)

In [ ]:
importances = rf.feature_importances_

features = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

features

Calcul ROC pour Random Forest

In [ ]:
y_prob = rf.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

roc_auc = auc(fpr, tpr)

Affichage ROC

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')

plt.plot([0, 1], [0, 1], linestyle='--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.title('ROC Curve - Random Forest')

plt.legend()

plt.savefig('../images/roc_curve.png')

plt.show()

Corrélations

Heatmap

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(df.corr(), cmap='coolwarm')

plt.title("Correlation Matrix")

plt.savefig('../images/correlation_matrix.png')

plt.show()

Feature Importance Chart

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

plt.figure(figsize=(10,6))

sns.barplot(
    data=feature_importance,
    x='Importance',
    y='Feature'
)

plt.title("Feature Importance - Random Forest")

plt.tight_layout()
plt.savefig('../images/feature_importance.png')

plt.show()

Validation croisée

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
scores = cross_val_score(
    rf,
    X,
    y,
    cv=5
)

print("CV Scores:", scores)
print("Mean Score:", scores.mean())
print("Std Dev:", scores.std())